# 📐 06 · 难度调整算法

> 比特币如何像恒温器一样自动维持 ~10 分钟出块。

## 6.1 难度调整公式

每 **2016 个区块**（约 2 周）调整一次：

```
new_difficulty = old_difficulty × (2016 × 600s) / actual_time_for_2016_blocks
```

**限制条件：** 单次调整不能超过 **×4** 或 **÷4**。

如果不用限制，极端事件（如中国禁令导致算力一夜跌 50%）会让难度瞬间崩塌。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

class DifficultyController:
    def __init__(self, target_block_time: float = 600.0):
        self.target_time = target_block_time
        self.interval = 2016
        self.difficulty = 1.0
        self.block_times = deque(maxlen=self.interval)
        self.history = [1.0]

    def mine_block(self, hashrate: float) -> float:
        expected = (self.difficulty * 2**32) / hashrate
        actual = random.expovariate(1.0 / expected)
        self.block_times.append(actual)

        if len(self.block_times) == self.interval:
            self._adjust()
            self.block_times.clear()
        return actual

    def _adjust(self):
        actual_avg = np.mean(self.block_times)
        target_total = self.target_time * self.interval
        ratio = target_total / (actual_avg * self.interval)
        ratio = max(0.25, min(4.0, ratio))  # clip to +/- 4x
        self.difficulty *= ratio
        self.history.append(self.difficulty)

print("✅ DifficultyController ready")


In [ ]:
# Three scenarios
scenarios = {
    "Exponential growth (BTC history)": 
        lambda i: 1e9 * (1.00003 ** i),
    "Sudden drop (China ban 2021)": 
        lambda i: 1e9 if i < 5000 else 0.5e9,
    "Recovery & surge": 
        lambda i: 1e9 * (0.2 + 0.8 * min(1, (i-3000)/7000)) if i > 3000 else 1e9,
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (title, hr_fn) in zip(axes, scenarios.items()):
    ctrl = DifficultyController()
    for i in range(20160):  # 10 adjustment periods
        hr = hr_fn(i)
        ctrl.mine_block(hr)
    
    ax.plot(ctrl.history, "o-", markersize=2, linewidth=1)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Adjustment period (×2016 blocks)")
    ax.set_ylabel("Difficulty")
    ax.grid(True, alpha=0.3)

plt.suptitle("Difficulty Retargeting — 3 Scenarios", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/difficulty_scenarios.png", dpi=100, bbox_inches="tight")
print("✅ Saved to data/difficulty_scenarios.png")
plt.close("all")


## 6.2 为什么是 2016？为什么是 4x？

**2016 个块 = 约 2 周：**
- 太短 → 对短期算力波动过度反应
- 太长 → 对算力趋势响应太慢
- 2 周是 Satoshi 经验选的平衡点

**4x 限制：**
- 防极端事件（如 2021 中国禁令）
- 即使全网算力跌 90%，难度一次最多跌到 1/4
- 代价：恢复期更长（要多等几个调整周期）

2021 年中国禁令后，难度连续 4 次下调（历时 2 个月），共降了约 50%。

## 6.3 真实难度历史

比特币难度从 2009 年至今：

```
2009:  难度 1           (CPU 挖矿)
2010:  难度 ~100         (GPU 兴起)
2013:  难度 ~500,000     (ASIC 降临)
2017:  难度 ~1,000,000,000,000
2021:  难度 ~25,000,000,000,000 → 跌至 14T (禁令)
2024:  难度 ~70,000,000,000,000
```

**70 万亿倍的增长**——超越任何摩尔定律能解释的速度。

## 小结

✅ 每 2016 块调整难度，维持 ~10 分钟出块
✅ ±4x 限制防止极端波动
✅ 难度只跟算力走，不跟价格走
✅ 算力是比特币的"安全预算"——越多人挖越安全
✅ 2021 禁令是最大单次难度下调 (-28%)

**下一课**: 挖矿经济学 & 攻击模型。